# MEN 4338 · Week 11 — Trends: simple and universal kriging of porosity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W11_trend_simple_universal_kriging.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Ordinary kriging assumes the mean is constant over the neighbourhood. Many variables have a **trend**:
porosity decreases with burial depth, temperature increases towards a heat source, the thickness of a
weathering profile follows the topography. This week, a petroleum case: porosity measured in 48 wells of a
4 km × 3 km sandstone reservoir (synthetic), tighter towards the north-east where the reservoir is deeper.

**What you will practise:** fitting a linear trend · residuals and their variogram · simple kriging (known
mean) · universal kriging (kriging with a drift) · the risk of extrapolating a trend.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/porosity_wells.csv', 'porosity_wells.csv'), ('tests/w11/q1.py', 'tests/q1.py'), ('tests/w11/q2.py', 'tests/q2.py'), ('tests/w11/q3.py', 'tests/q3.py'), ('tests/w11/q4.py', 'tests/q4.py'), ('tests/w11/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def exp_variogram(x, y, v, lag, nlags, azimuth=None, atol=22.5):
    """Experimental semivariogram. Lag class k (1..nlags) holds the pairs with |d - k*lag| < lag/2.
    With an azimuth (degrees clockwise from north), only pairs within +/- atol of that direction are kept.
    Returns mean distance, semivariance and number of pairs of each class (NaN where a class is empty)."""
    x, y, v = (np.asarray(a, float) for a in (x, y, v))
    i, j = np.triu_indices(len(v), 1)
    dx, dy = x[j] - x[i], y[j] - y[i]
    d = np.hypot(dx, dy)
    keep = np.ones(len(d), bool)
    if azimuth is not None:
        az = np.degrees(np.arctan2(dx, dy)) % 180
        diff = np.abs(az - azimuth % 180)
        keep = np.minimum(diff, 180 - diff) <= atol
    sq = 0.5 * (v[j] - v[i]) ** 2
    hm, g, n = np.full(nlags, np.nan), np.full(nlags, np.nan), np.zeros(nlags, int)
    for k in range(1, nlags + 1):
        m = keep & (np.abs(d - k * lag) < lag / 2)
        n[k - 1] = m.sum()
        if n[k - 1]:
            hm[k - 1], g[k - 1] = d[m].mean(), sq[m].mean()
    return hm, g, n

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

def ok_point(x0, y0, x, y, v, gamma, nmax=16):
    """Ordinary kriging at (x0, y0) with the nmax nearest data.
    gamma(dx, dy) is the variogram model, a function of the separation vector.
    Returns (estimate, kriging variance)."""
    d = np.hypot(x - x0, y - y0)
    idx = np.argsort(d)[:nmax]
    xs, ys, vs = x[idx], y[idx], v[idx]
    n = len(idx)
    A = np.ones((n + 1, n + 1)); A[n, n] = 0
    A[:n, :n] = gamma(xs[:, None] - xs[None, :], ys[:, None] - ys[None, :])
    b = np.ones(n + 1); b[:n] = gamma(xs - x0, ys - y0)
    sol = np.linalg.solve(A, b)
    w, mu = sol[:n], sol[n]
    return float(w @ vs), float(w @ b[:n] + mu)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

wells = pd.read_csv("porosity_wells.csv")
x, y = wells["x"].to_numpy(float), wells["y"].to_numpy(float)
phi_pct = 100 * wells["phi"].to_numpy()          # porosity in %
sc = plt.scatter(x, y, c=phi_pct, cmap="viridis"); plt.colorbar(sc, label="porosity (%)")
plt.gca().set_aspect("equal"); plt.xlabel("East (m)"); plt.ylabel("North (m)"); plt.show()

## 1. A variogram that never stops rising

With a trend, pairs far apart differ systematically, so the variogram of the raw data keeps increasing,
often faster than linearly. That is the signal to model the trend first:

$$Z(\mathbf{x}) = m(\mathbf{x}) + R(\mathbf{x}), \qquad m(\mathbf{x}) = a_0 + a_1 x + a_2 y$$

The residuals $R$ have a mean of zero and a variogram with a sill. Here the residual variogram is modelled by
`gamma_res`: nugget 0.3 + spherical 1.5, range 1000 m, isotropic (total sill 1.8 %²).

In [ ]:
def gamma_res(dx, dy):
    return spherical(np.hypot(dx, dy), 0.3, 1.5, 1000)

## 2. Three kinds of kriging

| | mean | system | use |
|---|---|---|---|
| **simple** (SK) | known (here: 0 for the residuals) | $C\lambda = c_0$, no constraint | residuals, normal scores, simulation |
| **ordinary** (OK) | unknown, constant locally | $\sum\lambda = 1$ | most grade estimation |
| **universal** (UK) | unknown, $a_0 + a_1x + a_2y$ | one constraint per drift term | trends |

SK uses covariances $C(h) = C(0) - \gamma(h)$; its variance is $C(0) - \sum\lambda_i c_{i0}$.
For UK with a linear drift the system is

$$
\begin{bmatrix} \Gamma & F \\ F^T & 0 \end{bmatrix}
\begin{bmatrix} \lambda \\ \mu \end{bmatrix} =
\begin{bmatrix} \gamma_0 \\ f_0 \end{bmatrix}, \qquad F = [1, x_i, y_i], \; f_0 = [1, x_0, y_0]
$$

(coordinates in km to keep the numbers balanced), and $\sigma^2_{UK} = [\lambda, \mu] \cdot [\gamma_0, f_0]$.

---
## Exercises

### Q1. The trend

Build `F = np.c_[np.ones(n), x / 1000, y / 1000]` and fit the trend by least squares: store the 3 coefficients
in `trend_coef`, the residuals in `resid`, and the trend value at the centre (2000, 1500) in `trend_at_center`.

In [ ]:
F = ...
trend_coef = ...
resid = ...
trend_at_center = ...
trend_coef

In [ ]:
grader.check("q1")

### Q2. Raw and residual variograms

Store the variances `raw_var` and `resid_var`, and the semivariances `g_raw` and `g_res` (lag 400 m,
8 lags). Plot both with the model `gamma_res`.

In [ ]:
raw_var = ...
resid_var = ...
g_raw = ...
g_res = ...

In [ ]:
grader.check("q2")

### Q3. Simple kriging of the residuals

Write `sk_residual(x0, y0, x, y, r, gamma, sill)` that krigs the residuals with a known mean of 0, using all
wells, and returns `(estimate, variance)`. Store `phi_sk_center` = trend + kriged residual at (2000, 1500).

In [ ]:
def sk_residual(x0, y0, x, y, r, gamma, sill):
    ...

phi_sk_center = ...
phi_sk_center

In [ ]:
grader.check("q3")

### Q4. Universal kriging

Write `uk_point(x0, y0, x, y, v, gamma)` with a linear drift (coordinates divided by 1000 in the drift),
using all wells, returning `(estimate, variance)`. Store the estimate at (2000, 1500) in `phi_uk_center`.
Compare with `phi_sk_center`.

In [ ]:
def uk_point(x0, y0, x, y, v, gamma):
    ...

phi_uk_center = ...
phi_uk_center

In [ ]:
grader.check("q4")

### Q5. Maps and extrapolation

Make `uk_map` on a 100 m grid (30 rows × 40 columns, node `(r, c)` at `x = 50 + 100*c`, `y = 50 + 100*r`) and
draw it. Then estimate the point (4400, 3300), **outside** the drilled area, with OK (`ok_point` with
`gamma_res` and `nmax=48`: `phi_ok_corner`) and with UK (`phi_uk_corner`). Which one would you trust for a
new well location there, and why is extrapolating the trend risky?

In [ ]:
uk_map = ...
phi_ok_corner = ...
phi_uk_corner = ...
phi_ok_corner, phi_uk_corner

In [ ]:
grader.check("q5")

### Going further (not checked)

* Pore volume: with a net thickness of 20 m, sum `uk_map / 100 * 100 * 100 * 20` over the map (m³).
* Fit a quadratic trend (add x², xy, y²). Do the residuals look better? Is the model still sensible at the edges?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()